# CLIP: 이미지와 텍스트를 같은 공간에서 비교하기

CLIP(Contrastive Language-Image Pre-training)은 이미지와 텍스트를 각각 인코딩한 뒤 같은 임베딩 공간에서 비교하도록 학습한 모델이다. 클래스별 학습 데이터를 다시 모으지 않아도 자연어 후보를 이용해 분류·검색할 수 있다는 점이 필요성이다.

멀티모달 검색, zero-shot 분류, 생성 모델의 결과 평가와 조건 선택에 사용되지만, CLIP 자체는 이미지나 문장을 생성하는 모델과 구별된다.

## dual encoder와 대조학습

CLIP은 image encoder와 text encoder를 따로 사용한다.

이미지 인코더의 공간 feature map이나 ViT token을 pooling과 projection으로 하나의 벡터로 바꾸고, <br>
텍스트 인코더의 token 표현도 pooling과 projection으로 하나의 벡터로 바꾼다.

두 전역 벡터를 L2 정규화하면 내적이 cosine similarity가 되어 서로 다른 modality를 같은 기준으로 비교할 수 있다.

학습 배치에 이미지-문장 쌍이 `N`개 있으면 모든 이미지와 모든 문장을 비교해 `(N, N)` 점수 행렬을 만든다. 정답 쌍은 대각선에 놓인다. 이미지에서 정답 문장을 찾는 cross entropy와 텍스트에서 정답 이미지를 찾는 cross entropy를 평균한 **대칭 대조 손실**이 정답 쌍은 가깝게, 나머지 쌍은 멀어지게 만든다.

### 핵심 용어
- **Feature map**: CNN이 추출한 채널과 공간 위치별 이미지 특징
- **ViT token**: 이미지 patch를 Transformer가 처리할 벡터로 변환한 표현
- **Pooling**: 여러 위치의 feature나 token을 하나의 전역 벡터로 요약하는 연산
- **Projection**: 서로 다른 encoder 출력을 같은 차원의 embedding 공간으로 변환하는 학습 가능한 계층
- **L2 normalization**: 벡터의 길이가 1이 되도록 각 값을 조정하는 연산
- **Modality**: 이미지와 텍스트처럼 서로 다른 형태의 데이터 유형
- **Score matrix**: 배치의 모든 이미지와 텍스트 조합에 대한 유사도를 담은 `(N, N)` 행렬
- **Symmetric contrastive loss**: 이미지→텍스트와 텍스트→이미지 cross entropy를 평균한 손실

### CLIP 구조 그림 해석

![OpenAI CLIP의 학습과 zero-shot 추론 구조](https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png)

그림은 왼쪽의 **학습 과정**과 오른쪽의 **분류 과정**으로 나누어 읽는다.

#### 1. 왼쪽: 이미지와 문장의 관계 학습

CLIP은 강아지 이미지와 '강아지 사진'처럼 서로 맞는 이미지·문장을 각각 벡터로 변환한다. 맞는 쌍의 벡터는 가깝게, 틀린 쌍의 벡터는 멀어지도록 학습한다.

#### 2. 오른쪽: 분류할 후보 문장 준비

예측 후보가 고양이, 강아지, 자동차라면 각 label을 자연어 문장으로 바꾼 뒤 text encoder로 벡터화한다.

```text
고양이 → "a photo of a cat" → 텍스트 벡터
강아지 → "a photo of a dog" → 텍스트 벡터
자동차 → "a photo of a car" → 텍스트 벡터
```

#### 3. 오른쪽: 새 이미지 분류

새 이미지를 이미지 벡터로 변환하고 후보 text vector들과의 유사도를 비교한다.

```text
고양이: 0.12
강아지: 0.87  ← 가장 유사한 후보
자동차: 0.03
```

가장 높은 점수를 받은 `강아지`가 최종 예측이 된다. 강아지 분류 모델을 별도로 다시 학습하지 않고 자연어 후보만으로 분류하므로 **zero-shot 분류**라고 한다.

> **핵심**: CLIP은 정답 label을 직접 만드는 모델이 아니라, 이미지와 주어진 후보 문장 중 가장 잘 어울리는 조합을 찾는 모델이다.

출처: [OpenAI CLIP 저장소](https://github.com/openai/CLIP), 라이선스: [MIT License](https://github.com/openai/CLIP/blob/main/LICENSE)이다.


## 정규화된 임베딩과 similarity matrix

입력은 이미지 벡터 `I`개와 텍스트 후보 벡터 `M`개이다.

각 벡터를 L2 정규화하고 행렬곱하면 출력은 `(I, M)` cosine similarity matrix가 된다. <br>
행은 이미지, 열은 텍스트 후보이므로 각 행의 `argmax`가 해당 이미지에 가장 가까운 후보의 인덱스이다.

아래 2차원 값은 계산 원리를 빠르게 확인하기 위해 직접 만든 **toy embedding**이다. 학습된 CLIP encoder의 activation이나 의미 특징으로 해석하지 않으며, 다음 실제 모델 코드에서 동일한 shape 흐름만 연결한다.

In [1]:
import torch
import torch.nn.functional as F

# 출력하는 소수점 자릿수를 4로 지정, 실제 계산에는 영향 X
torch.set_printoptions(precision=4, sci_mode=False)

image_vectors = torch.tensor([[2.0, 1.0], [0.0, 3.0]])

text_vectors = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

text_candidates = ["horizontal pattern", "vertical pattern", "diagonal pattern"]

# 정규화 과정 (벡터 길이(벡터 데이터들 간의 거리)를 1로 만듦)
image_embeddings = F.normalize(image_vectors, dim=1)
text_embeddings = F.normalize(text_vectors, dim=1)
similarity = image_embeddings @ text_embeddings.T

# 유사도가 가장 높은 인덱스 찾기
best_text_indices = similarity.argmax(dim=1)

best_texts = [text_candidates[index] for index in best_text_indices.tolist()]

print("image embedding shape:", tuple(image_embeddings.shape))
print("text embedding shape:", tuple(text_embeddings.shape))
print("image norms:", image_embeddings.norm(dim=1))
print("similarity shape:", tuple(similarity.shape))
print(similarity)
print("row-wise argmax:", best_text_indices.tolist())
print("selected texts:", best_texts)

image embedding shape: (2, 2)
text embedding shape: (3, 2)
image norms: tensor([1.0000, 1.0000])
similarity shape: (2, 3)
tensor([[0.8944, 0.4472, 0.9487],
        [0.0000, 1.0000, 0.7071]])
row-wise argmax: [2, 1]
selected texts: ['diagonal pattern', 'vertical pattern']


### scaled logits, 상대 확률과 대칭 대조 손실
CLIP은 정규화된 유사도에 학습된 양의 inverse-temperature scale을 곱해 logits_per_image를 만든다.

Hugging Face CLIPModel의 model.logit_scale은 이 양수를 직접 저장하지 않고 로그 영역의 raw parameter를 저장한다. 실제 forward에서는 model.logit_scale.exp()를 적용한 양수가 inverse temperature로 사용된다. 이 값이 커질수록 temperature는 낮아지고 높은 점수와 낮은 점수의 차이가 더 선명해진다. softmax(dim=1)은 한 이미지가 현재 텍스트 후보들 중 어디에 더 가까운지를 나타내는 상대값을 만든다. 이 값은 후보 집합이 바뀌면 함께 바뀌므로 보정된 절대 확률로 볼 수 없다.

학습에서는 이미지 i와 같은 인덱스의 텍스트 i가 정답이라고 둔다. (N,N) logits와 전치 logits에 각각 cross entropy를 적용하고 평균하면 이미지→텍스트와 텍스트→이미지 방향을 모두 학습할 수 있다. 아래 값도 손실 계산 구조만 확인하는 결정적 toy 데이터이다.

In [2]:
paired_image_embeddings = F.normalize(
    torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]), dim=1
)
paired_text_embeddings = F.normalize(
    torch.tensor([[1.0, 0.1], [0.1, 1.0], [1.0, 0.9]]), dim=1
)

inverse_temperature = torch.tensor(4.0)
scaled_logits = inverse_temperature * (paired_image_embeddings @ paired_text_embeddings.T)
probabilities = scaled_logits.softmax(dim=1)

targets = torch.arange(len(paired_image_embeddings))
image_to_text_loss = F.cross_entropy(scaled_logits, targets)
text_to_image_loss = F.cross_entropy(scaled_logits.T, targets)
symmetric_loss = (image_to_text_loss + text_to_image_loss) / 2

print("scaled logits shape:", tuple(scaled_logits.shape))
print(scaled_logits)
print("row probabilities:")
print(probabilities)
print("row probability sums:", probabilities.sum(dim=1))
print("row-wise argmax:", probabilities.argmax(dim=1).tolist())
print("symmetric contrastive loss:", round(symmetric_loss.item(), 4))

scaled logits shape: (3, 3)
tensor([[3.9801, 0.3980, 2.9732],
        [0.3980, 3.9801, 2.6759],
        [3.0958, 3.0958, 3.9945]])
row probabilities:
tensor([[0.7178, 0.0200, 0.2622],
        [0.0214, 0.7697, 0.2089],
        [0.2244, 0.2244, 0.5512]])
row probability sums: tensor([1.0000, 1.0000, 1.0000])
row-wise argmax: [0, 1, 2]
symmetric contrastive loss: 0.4011


## feature map, token과 CLIP embedding의 경계

CNN backbone의 중간 출력은 보통 `(B, C, H, W)` feature map이다. `H×W` 위치 정보가 남아 있으므로 한 채널이나 한 위치를 곧바로 CLIP embedding이라고 부르지 않는다. global pooling과 projection을 거친 뒤에야 이미지마다 하나의 `(B, D)` 전역 임베딩이 된다.

ViT backbone의 중간 출력은 보통 `(B, L, D_v)` token sequence이다. `L`은 CLS token을 포함한 visual token 수이다. CLS 또는 pooled representation과 projection을 거친 `(B, D)`가 CLIP의 비교용 이미지 임베딩이다. 텍스트 인코더의 중간 출력은 후보 수 `M`, 문장별 token 수 `L_t`를 사용해 `(M, L_t, D_t)`로 나타낼 수 있다. token sequence 전체가 아니라 pooled representation과 projection을 거친 `(M, D)` 전역 임베딩을 비교에 사용한다.

따라서 실제 추론에서 `image_embeds`는 `(I, D)`, `text_embeds`는 `(M, D)`, `logits_per_image`는 `(I, M)`이 된다. 여기서 `I`는 이미지 수, `M`은 텍스트 후보 수이다. CLIP에는 고정된 클래스 개수의 classifier head 대신 자연어 후보 임베딩이 그 역할을 한다.

## 실제 Transformers 추론 준비

실제 경로는 공개 체크포인트 `openai/clip-vit-base-patch32`를 사용한다. 첫 실행에는 모델 가중치 약 605MB와 processor 설정을 내려받으므로 네트워크와 디스크 공간이 필요하다. 공개 체크포인트는 별도 인증 정보 없이 내려받을 수 있으며 CPU에서도 추론할 수 있다. GPU가 있으면 자동으로 사용한다.

현재 PyTorch 중심 실습 환경에서는 첫 `transformers` import보다 먼저 `USE_TF=0`을 설정해 불필요한 TensorFlow 로드를 막는다. 이미 `transformers`를 import한 커널이라면 커널을 다시 시작하고 아래 셀부터 실행한다. 입력 이미지는 앞에서 본 OpenAI 공식 구조 이미지 URL이며 후보 문장은 이미지의 내용을 포함한 네 가지 prompt이다.

In [3]:
import os

os.environ["USE_TF"] = "0"

import torch
from io import BytesIO
import requests
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
IMAGE_URL = "https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png"
TEXT_CANDIDATES = [
    "a diagram explaining contrastive image-text learning",
    "a photo of a dog",
    "a photo of a kitchen",
    "a photo of a bicycle",
]
print("model id:", MODEL_ID)
print("candidate count:", len(TEXT_CANDIDATES))
print("input image:", IMAGE_URL)

model id: openai/clip-vit-base-patch32
candidate count: 4
input image: https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png


## processor와 model 로드

입력 `MODEL_ID`는 Hugging Face Hub의 공개 체크포인트를 가리킨다. `CLIPProcessor.from_pretrained`는 이미지 전처리기와 tokenizer 설정을 가져오고, `CLIPModel.from_pretrained`는 모델 구조와 가중치를 가져온다. 출력 `processor`와 `model`은 다음 셀에서 같은 이미지·문장 배치를 만드는 데 사용한다.

`model.eval()`은 dropout 등 학습 동작을 추론 모드로 전환한다. 최초 실행에는 모델 다운로드를 위한 네트워크가 필요하다. 다운로드가 끝나면 최종 device가 출력되는지 확인한다.

In [4]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("inference device:", device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

inference device: cpu


## 이미지와 텍스트를 하나의 배치로 변환하기

입력 이미지 URL은 `requests.get`의 `timeout=30`으로 내려받고 `raise_for_status()`로 HTTP 실패를 즉시 확인한다. PIL 이미지를 RGB로 통일한 뒤 processor에 전달한다. `text`는 후보 목록, `images`는 PIL 이미지, `padding=True`는 문장 길이 맞춤, `truncation=True`는 CLIP 최대 길이 초과 방지, `return_tensors="pt"`는 PyTorch Tensor 출력을 뜻한다.

processor 출력의 `pixel_values`는 이미지 배치 `(I, 3, 224, 224)`이고, `input_ids`와 `attention_mask`의 첫 축은 후보 수 `M`이다. 각 Tensor를 model과 같은 device로 옮긴 `inputs`가 다음 forward 입력이 된다.

In [6]:
response = requests.get(IMAGE_URL, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")

inputs = processor(
    text=TEXT_CANDIDATES,
    images=image,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
print("original image size:", image.size)
print("pixel_values shape:", tuple(inputs["pixel_values"].shape))
print("input_ids shape:", tuple(inputs["input_ids"].shape))
print("attention_mask shape:", tuple(inputs["attention_mask"].shape))

original image size: (2162, 762)
pixel_values shape: (1, 3, 224, 224)
input_ids shape: (4, 11)
attention_mask shape: (4, 11)


## model forward에서 zero-shot 예측까지

`torch.inference_mode()`는 gradient 기록을 끄고 추론 메모리를 줄인다. 입력 dictionary를 keyword argument로 풀어 `model`에 전달하면 `image_embeds`는 `(I,D)`, `text_embeds`는 `(M,D)`, `logits_per_image`는 `(I,M)`으로 나온다. logits의 텍스트 후보 축에 `softmax(dim=1)`을 적용하고 행별 `argmax`를 구하면 이미지마다 가장 높은 후보를 선택할 수 있다.

출력 probability는 네 후보 안에서만 합이 1인 상대값이다. 후보를 추가·삭제하거나 표현을 바꾸면 값과 예측이 달라질 수 있으며 현실 전체에 대한 보정된 절대 확률로 해석하지 않는다.

In [7]:
with torch.inference_mode():
    outputs = model(**inputs)
    probabilities = outputs.logits_per_image.softmax(dim=1)

predicted_indices = probabilities.argmax(dim=1).cpu().tolist()
predicted_labels = [TEXT_CANDIDATES[index] for index in predicted_indices]
print("image_embeds shape:", tuple(outputs.image_embeds.shape))
print("text_embeds shape:", tuple(outputs.text_embeds.shape))
print("logits_per_image shape:", tuple(outputs.logits_per_image.shape))
print("candidate-relative probabilities:", probabilities.cpu())
print("predicted labels:", predicted_labels)

image_embeds shape: (1, 512)
text_embeds shape: (4, 512)
logits_per_image shape: (1, 4)
candidate-relative probabilities: tensor([[0.9999, 0.0001, 0.0000, 0.0000]])
predicted labels: ['a diagram explaining contrastive image-text learning']


## prompt, 후보 집합과 모델 한계

zero-shot 분류의 클래스는 고정 classifier weight가 아니라 prompt 임베딩으로 만들어진다. `dog`보다 `a photo of a dog`처럼 학습 데이터의 자연어 설명에 가까운 표현이 유리할 수 있다. 단, 좋은 template은 데이터와 과제에 따라 달라지므로 여러 template의 임베딩이나 점수를 비교하고 후보 목록에 필요한 클래스를 빠뜨리지 않아야 한다.

CLIP은 영어 중심 데이터와 웹 이미지-텍스트 쌍에서 학습되어 언어·문화·직업·인구집단 편향을 포함할 수 있다. 세밀한 속성 구분, 객체 수 세기, 학습 분포 밖 이미지, 후보 taxonomy가 부정확한 상황에서도 성능이 낮아질 수 있다. 얼굴 식별이나 사람에 관한 고위험 판단에 바로 사용하지 않고 과제별 데이터로 성능과 공정성을 별도로 평가한다.

## 핵심 정리와 다음 연결

CLIP의 비교 흐름은 `encoder 중간 표현 → pooling·projection → 정규화된 전역 embedding → scaled similarity → 후보 집합 내 softmax → argmax`이다. CNN의 feature map과 ViT의 token sequence는 이 흐름의 중간 표현이고, CLIP이 서로 비교하는 값은 projection 뒤의 전역 임베딩이다. 이 경계를 기억하면 다음 ViT·CLIP·BLIP 단원에서 backbone 표현, 멀티모달 정렬, 생성 decoder의 역할을 혼동하지 않을 수 있다.